# DDPM Training on FFHQ 128×128

**Архитектура:** UNet с Sinusoidal Time Embedding, ResNet блоками и Self-Attention  
**Loss:** `MSE(predicted_noise, real_noise)` — простейший вариант из статьи Ho et al., 2020  
**Schedule:** Cosine β-schedule (лучше чем linear для лиц)  
**EMA:** Exponential Moving Average весов → значительно улучшает качество  

> ⏳ **DDPM обучается медленно.** Один шаг инференса = T=1000 forward-пассов UNet.  
> `sample_every=20` — разумный компромисс.

---

## ⚙️ Конфигурация

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

DATA_ROOT    = '../data'
OUTPUT_DIR   = '../checkpoints/ddpm'

BASE_CHANNELS = 64     # каналы UNet (bottleneck = base_ch * 8 = 512)
TIME_EMB_DIM  = 256    # размерность time embedding
TIMESTEPS     = 1000   # T — количество шагов диффузии
SCHEDULE      = 'cosine'  # 'cosine' | 'linear'
DROPOUT       = 0.1

EPOCHS        = 200
BATCH_SIZE    = 16     # DDPM требует больше памяти
LR            = 2e-4
GRAD_CLIP     = 1.0
WARMUP_EPOCHS = 5
EMA_DECAY     = 0.9999  # 0 = отключить EMA

SAVE_EVERY    = 10
SAMPLE_EVERY  = 20     # медленный семплинг — реже
LOG_EVERY     = 10
N_SAMPLES     = 16     # меньше из-за T=1000 шагов

DRY_RUN       = False
RESUME        = None

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 📂 Загрузка данных

In [ ]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

if DRY_RUN:
    class _Dummy:
        def __init__(self): self.n = 10
        def __len__(self): return self.n
        def __iter__(self):
            for _ in range(self.n): yield torch.randn(BATCH_SIZE, 3, 128, 128)
    train_loader = _Dummy()
    print('⚠️  DRY RUN')
else:
    from data.dataset import get_dataloaders
    train_loader, _ = get_dataloaders(
        data_root=DATA_ROOT, batch_size=BATCH_SIZE, num_workers=4,
    )

print(f'Train батчей: {len(train_loader)}')

plt.rcParams.update({'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
                     'text.color': '#c9d1d9', 'axes.titlecolor': '#c9d1d9'})

sample_batch = next(iter(train_loader))[:16]
grid = make_grid((sample_batch + 1) / 2, nrow=8, padding=2, pad_value=0.1)
fig, ax = plt.subplots(figsize=(14, 4))
ax.imshow(grid.permute(1, 2, 0).numpy())
ax.axis('off')
ax.set_title('FFHQ — Real Images (preview)', fontsize=11)
plt.tight_layout()
plt.show()

## 📈 Визуализация β-расписания

In [ ]:
# Сравнение linear vs cosine schedule
from DDPM.ddpm import get_beta_schedule
import matplotlib.pyplot as plt
import torch

T     = TIMESTEPS
lin   = get_beta_schedule(T, 'linear')
cos   = get_beta_schedule(T, 'cosine')
lin_a = torch.cumprod(1 - lin, dim=0)
cos_a = torch.cumprod(1 - cos, dim=0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('β-Schedule: Linear vs Cosine', fontsize=12, color='#58a6ff')
fig.patch.set_facecolor('#0d1117')

for ax in axes:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='#8b949e')
    ax.spines[:].set_color('#30363d')

axes[0].plot(lin.numpy(),   color='#f78166', label='Linear β')
axes[0].plot(cos.numpy(),   color='#58a6ff', label='Cosine β')
axes[0].set_title('β_t (noise schedule)', color='#c9d1d9')
axes[0].legend()
axes[0].grid(color='#21262d', linestyle='--', alpha=0.5)

axes[1].plot(lin_a.numpy(), color='#f78166', label='Linear ᾱ_t')
axes[1].plot(cos_a.numpy(), color='#58a6ff', label='Cosine ᾱ_t')
axes[1].set_title('ᾱ_t = ∏(1-β_t) (signal retention)', color='#c9d1d9')
axes[1].legend()
axes[1].grid(color='#21262d', linestyle='--', alpha=0.5)

plt.tight_layout()
os.makedirs(OUTPUT_DIR + '/plots', exist_ok=True)
plt.savefig(f'{OUTPUT_DIR}/plots/beta_schedule.png', dpi=120,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 🔊 Визуализация forward process (зашумление)

In [ ]:
# Показываем как x_0 превращается в шум за T шагов
from DDPM.ddpm import UNet, DDPM as DDPMClass

_dummy_unet = UNet(3, BASE_CHANNELS, TIME_EMB_DIM, DROPOUT).to(DEVICE)
_ddpm_temp  = DDPMClass(_dummy_unet, TIMESTEPS, SCHEDULE, str(DEVICE))

x0 = next(iter(train_loader))[:1].to(DEVICE)
show_ts = [0, 100, 250, 500, 750, 999]

fig, axes = plt.subplots(1, len(show_ts), figsize=(14, 3))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('Forward Diffusion Process: x_0 → x_T', color='#58a6ff', fontsize=11)

with torch.no_grad():
    for ax, t_val in zip(axes, show_ts):
        t_tensor = torch.tensor([t_val], device=DEVICE)
        x_t, _   = _ddpm_temp.q_sample(x0, t_tensor)
        img      = ((x_t[0].cpu() + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f't={t_val}', color='#8b949e', fontsize=9)
        ax.set_facecolor('#0d1117')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plots/forward_diffusion.png', dpi=120,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
del _dummy_unet, _ddpm_temp

## 🏗️ Модель

In [ ]:
import copy
from DDPM.ddpm import UNet, DDPM as DDPMClass
from utils.utils import print_model_info

unet = UNet(
    img_channels=3,
    base_channels=BASE_CHANNELS,
    time_emb_dim=TIME_EMB_DIM,
    dropout=DROPOUT,
).to(DEVICE)

ddpm = DDPMClass(unet, TIMESTEPS, SCHEDULE, str(DEVICE))

print_model_info(unet, f'UNet DDPM  (base_ch={BASE_CHANNELS}, T={TIMESTEPS}, {SCHEDULE})')

# Тест
with torch.no_grad():
    x_test  = torch.randn(2, 3, 128, 128, device=DEVICE)
    t_test  = torch.randint(0, TIMESTEPS, (2,), device=DEVICE)
    out     = unet(x_test, t_test)
    loss    = ddpm.loss_fn(x_test)
print(f'✓ Forward pass OK: out={out.shape}, loss={loss.item():.4f}')

## 🚀 Обучение

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from tqdm.notebook import tqdm

from utils.utils import (
    TrainingLogger, Visualizer,
    save_checkpoint, load_checkpoint, save_sample_grid,
)

torch.manual_seed(42)

optimizer = optim.AdamW(unet.parameters(), lr=LR)
warmup    = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=WARMUP_EPOCHS)
cosine    = CosineAnnealingLR(optimizer, T_max=max(1, EPOCHS - WARMUP_EPOCHS), eta_min=LR * 0.01)
scheduler = SequentialLR(optimizer, [warmup, cosine], milestones=[WARMUP_EPOCHS])

# EMA
ema_model = None
if EMA_DECAY > 0:
    ema_model = copy.deepcopy(unet).eval()
    for p in ema_model.parameters(): p.requires_grad_(False)
    print(f'EMA enabled (decay={EMA_DECAY})')

def update_ema(ema, model, decay):
    with torch.no_grad():
        for ep, mp in zip(ema.parameters(), model.parameters()):
            ep.data.mul_(decay).add_(mp.data, alpha=1-decay)

logger = TrainingLogger(OUTPUT_DIR, model_name='DDPM')
vis    = Visualizer(OUTPUT_DIR, model_name='DDPM', inline=True)

start_epoch = 1
best_loss   = float('inf')

if RESUME and os.path.exists(RESUME):
    ckpt = load_checkpoint(RESUME, device=str(DEVICE))
    unet.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    if ema_model and 'ema' in ckpt: ema_model.load_state_dict(ckpt['ema'])
    start_epoch = ckpt.get('epoch', 0) + 1

_epochs = 2 if DRY_RUN else EPOCHS

for epoch in range(start_epoch, _epochs + 1):
    unet.train()
    loss_acc = 0.0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch:3d}/{_epochs}', leave=False)
    for batch in pbar:
        imgs = batch.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        loss = ddpm.loss_fn(imgs)
        loss.backward()
        if GRAD_CLIP > 0:
            torch.nn.utils.clip_grad_norm_(unet.parameters(), GRAD_CLIP)
        optimizer.step()
        if ema_model: update_ema(ema_model, unet, EMA_DECAY)
        loss_acc += loss.item()
        pbar.set_postfix(mse=f'{loss.item():.5f}')

    scheduler.step()
    avg_loss = loss_acc / max(len(train_loader), 1)
    logger.log_epoch(epoch=epoch, mse_loss=avg_loss)

    is_best = avg_loss < best_loss
    if is_best: best_loss = avg_loss

    print(f'Epoch {epoch:3d}/{_epochs}  |  '
          f'mse_loss={avg_loss:.5f}  '
          f'lr={optimizer.param_groups[0]["lr"]:.2e}'
          + ('  ★ best' if is_best else ''))

    # ── Образцы ────────────────────────────────────────────────────────
    if epoch % SAMPLE_EVERY == 0 or epoch == _epochs:
        print(f'  🎨 Семплируем {N_SAMPLES} изображений (T={TIMESTEPS} шагов)...')
        _sm = ema_model if ema_model else unet
        _dd = DDPMClass(_sm, TIMESTEPS, SCHEDULE, str(DEVICE))
        samples = _dd.sample(n=N_SAMPLES, verbose=True)
        save_sample_grid(samples,
            path=f'{OUTPUT_DIR}/samples/gen_ep{epoch:03d}.png',
            nrow=4, title=f'DDPM Generated — Epoch {epoch}')

    # ── Чекпоинт ───────────────────────────────────────────────────────
    if epoch % SAVE_EVERY == 0 or epoch == _epochs:
        state = {'epoch': epoch, 'model': unet.state_dict(),
                 'optimizer': optimizer.state_dict(), 'best_loss': best_loss}
        if ema_model: state['ema'] = ema_model.state_dict()
        save_checkpoint(state, OUTPUT_DIR,
                        filename=f'checkpoint_ep{epoch:03d}.pt', is_best=is_best)

    # ── Live plot ──────────────────────────────────────────────────────
    if epoch % SAVE_EVERY == 0 or epoch == _epochs:
        vis.plot_curves(logger.epoch_history, epoch=epoch, save=True, show=True)

logger.close()
print(f'\n✅ Обучение завершено! Лучший mse={best_loss:.5f}')

## 📊 Финальные результаты

In [ ]:
print(f'Финальная генерация {N_SAMPLES} изображений...')
_sm = ema_model if ema_model else unet
_dd = DDPMClass(_sm, TIMESTEPS, SCHEDULE, str(DEVICE))
final_samples = _dd.sample(n=N_SAMPLES, verbose=True)

poster_path = vis.plot_final_summary(
    logger.epoch_history,
    samples=final_samples,
    extra_info=f'T={TIMESTEPS}, {SCHEDULE}',
)

from IPython.display import Image as IPyImage
IPyImage(poster_path)

In [ ]:
# ── Визуализация процесса денойзинга (обратный процесс) ───────────────
import matplotlib.pyplot as plt

_sm = ema_model if ema_model else unet
_sm.eval()
_dd = DDPMClass(_sm, TIMESTEPS, SCHEDULE, str(DEVICE))

# Сохраняем промежуточные шаги
show_at = [999, 800, 600, 400, 200, 100, 50, 0]
snapshots = {}

with torch.no_grad():
    x = torch.randn(1, 3, 128, 128, device=DEVICE)
    for t_idx in reversed(range(TIMESTEPS)):
        x = _dd.p_sample(x, t_idx)
        if t_idx in show_at:
            snapshots[t_idx] = x[0].cpu().clamp(-1, 1)

plt.rcParams.update({'figure.facecolor': '#0d1117'})
fig, axes = plt.subplots(1, len(show_at), figsize=(18, 3))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('DDPM: Reverse Denoising Process (xT → x0)', color='#58a6ff', fontsize=12)

for ax, t_val in zip(axes, sorted(snapshots.keys(), reverse=True)):
    img = ((snapshots[t_val] + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f't={t_val}', color='#8b949e', fontsize=9)
    ax.set_facecolor('#0d1117')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plots/denoising_steps.png', dpi=120,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()